# MMS-1B-all LoRA Fine-tuning for Maltese ASR — Retrain v2

**This is the retrain of the April MMS-1B-all notebook with the methodological fixes identified in the supervision review.**

## Changes from the previous version

1. **Proper train/val/test split.** The 4,481-utterance train CSV is split 90/10 into train (~4,033) and val (~448); the 498-utterance test CSV is held out. The val set drives early stopping and best-checkpoint selection. The test set is touched only for the zero-shot baseline and the final evaluation. Protocol: Mainzinger & Levow (2024, ACL SRW), the closest published analog (MMS adapter fine-tuning on very low-resource Mvskoke with explicit train/dev/test partitioning).

2. **Gradient flow fix.** `model.enable_input_require_grads()` is now called and `gradient_checkpointing=True` is retained. The previous version's contradiction — comment saying "gradient_checkpointing disabled" but actual setting `True` with no `enable_input_require_grads()` — was a silent gradient-flow bug that may have left the encoder LoRA adapters under-trained while learning was carried by `modules_to_save=["lm_head"]`. With `mask_time_prob=mask_feature_prob=0.0` the in-place masking conflict that motivated the earlier workaround does not fire, so the standard hook is safe.

3. **Epoch ceiling raised to 25, patience to 5.** Previous run's WER was still falling at the 15-epoch ceiling — model never converged. The longer ceiling lets early stopping (rather than the budget) terminate training.

4. **New save directory.** `MMS_1B_LoRA_Maltese_v2_proper_split` so the original checkpoint is preserved.

5. **Zero-shot baseline added.** Important here because MMS-1B-all already ships with a pretrained Maltese adapter — quantifying how much LoRA fine-tuning adds on top of that adapter is essential for the cross-model comparison.

## Hyperparameters (defended in thesis methodology)

- **LR = 1e-3, warmup = 100:** canonical MMS adapter fine-tuning configuration (von Platen 2023, HuggingFace MMS adapter blog), aligned with the original MMS paper (Pratap et al., 2024, JMLR). The same LR has been used successfully for MMS adapter fine-tuning in subsequent peer-reviewed work (Mainzinger & Levow 2024).
- **LoRA r=32, α=64, q/v targets, dropout=0.05:** held identical across all 3 models for fair architectural comparison.
- **`target_lang="mlt"` at load:** attaches the pretrained Maltese-specific lm_head and tokenizer vocabulary.
- **`modules_to_save=["lm_head"]`:** refines the MMS-pretrained Maltese head on the read-speech MASRI domain (MMS was trained on MMS-lab, mostly religious text). This is *refinement* of a good starting head, not random initialisation.
- **Patience=5 on val WER**, seed=42, identical text normalisation, FP16, gradient checkpointing.


## 1. Setup

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate jiwer librosa soundfile sentencepiece
!pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os
import torch
from transformers import set_seed


os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


drive.mount('/content/drive')

# Reproducibility — identical seed across all 3 model notebooks
set_seed(42)

# Paths
model_id = "facebook/mms-1b-all"
model_save_dir = "/content/drive/My Drive/ASRModels/MMS_1B_LoRA_Maltese_v2_proper_split"
train_csv_path = "/content/drive/My Drive/Thesis Project/MASRI_HEADSET_v2/train_metadata.csv"
test_csv_path  = "/content/drive/My Drive/Thesis Project/MASRI_HEADSET_v2/test_metadata.csv"

# Maltese language code in MMS taxonomy (ISO 639-3)
TGT_LANG = "mlt"

os.makedirs(model_save_dir, exist_ok=True)
print(f"Models will be saved to: {model_save_dir}")

Mounted at /content/drive
Models will be saved to: /content/drive/My Drive/ASRModels/MMS_1B_LoRA_Maltese_v2_proper_split


## 2. Data preparation

In [ ]:
from datasets import load_dataset, DatasetDict, Audio
import re

# Load all CSVs. The original train CSV becomes train+val; the test CSV is held out.
raw = load_dataset("csv", data_files={
    "trainval": train_csv_path,
    "test":     test_csv_path,
})


drive_base_path = "/content/drive/My Drive/Thesis Project/MASRI_HEADSET_v2/"

def fix_paths_bulletproof(batch):
    old_path = batch["file_path"].replace("\\", "/")
    if "/speech/" in old_path:
        relative_path = "speech/" + old_path.split("/speech/")[-1]
    else:
        relative_path = old_path.split("/")[-1]
    batch["file_path"] = os.path.join(drive_base_path, relative_path)
    return batch

raw = raw.map(fix_paths_bulletproof)

# 90/10 train/val split of the trainval portion (Mainzinger & Levow 2024 protocol).
# Seed matches the global seed so the split is deterministic across the 3 notebooks.
split = raw["trainval"].train_test_split(test_size=0.10, seed=42, shuffle=True)

masri_dataset = DatasetDict({
    "train": split["train"],
    "val":   split["test"],
    "test":  raw["test"],
})

print(f"Train samples: {len(masri_dataset['train'])}")
print(f"Val   samples: {len(masri_dataset['val'])}")
print(f"Test  samples: {len(masri_dataset['test'])}  (held out — touched only at zero-shot baseline and final evaluation)")


def clean_text(batch):
    text = batch["transcription"].lower()
    text = re.sub(r"[^\w\s'-]", "", text)
    batch["transcription"] = re.sub(r"\s+", " ", text).strip()
    return batch

masri_dataset = masri_dataset.map(clean_text)


masri_dataset = masri_dataset.cast_column("file_path", Audio(sampling_rate=16000))


Generating trainval split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4481 [00:00<?, ? examples/s]

Map:   0%|          | 0/498 [00:00<?, ? examples/s]

Train samples: 4032
Val   samples: 449
Test  samples: 498  (held out — touched only at zero-shot baseline and final evaluation)


Map:   0%|          | 0/4032 [00:00<?, ? examples/s]

Map:   0%|          | 0/449 [00:00<?, ? examples/s]

Map:   0%|          | 0/498 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoProcessor


processor = AutoProcessor.from_pretrained(model_id, target_lang=TGT_LANG)
print(f"MMS processor loaded with target_lang='{TGT_LANG}'")
print(f"Vocabulary size: {len(processor.tokenizer)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

MMS processor loaded with target_lang='mlt'
Vocabulary size: 83


In [ ]:
def prepare_dataset(batch):
    audio = batch["file_path"]
    batch["input_values"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

masri_dataset = masri_dataset.map(
    prepare_dataset,
    remove_columns=masri_dataset.column_names["train"],
    num_proc=8,
)

print("Dataset preprocessing complete.")

Map (num_proc=8):   0%|          | 0/4032 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/449 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/498 [00:00<?, ? examples/s]

Dataset preprocessing complete.


## 3. Data collator and metrics

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Union
from transformers import Wav2Vec2Processor
from jiwer import wer as compute_wer, cer as compute_cer
import numpy as np

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids":     f["labels"]}       for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(
            label_features, padding=self.padding, return_tensors="pt"
        )


        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)


def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s'-]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    pred_str  = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    # Filter empty references
    pairs = [(p, l) for p, l in zip(pred_str, label_str) if l.strip()]
    if not pairs:
        return {"wer": 1.0, "cer": 1.0}
    pred_str, label_str = zip(*pairs)

    return {
        "wer": compute_wer(list(label_str), list(pred_str)),
        "cer": compute_cer(list(label_str), list(pred_str)),
    }

## 4. Model and LoRA configuration

Key points:

1. **`target_lang="mlt"` + `ignore_mismatched_sizes=True`** is the canonical MMS loading pattern (HuggingFace MMS docs). This attaches the pretrained Maltese-specific lm_head and adapter weights — MMS's biggest advantage over models without Maltese pretraining for this task.

2. **`mask_time_prob = mask_feature_prob = 0.0`** disables SpecAugment-style masking. SpecAugment is a *pretraining* regularisation technique; for low-resource fine-tuning the HuggingFace XLS-R / Wav2Vec2 recipe sets these to 0 (von Platen 2021; 2023). With both at 0, the in-place `hidden_states[mask_indices] = ...` write inside `_mask_hidden_states` does not execute, so the previous notebook's masking-vs-leaf-grad workaround is unnecessary.

3. **`get_input_embeddings` override** points at `feature_projection.projection`. Audio models in the Wav2Vec2 family have no token-input embeddings, so we point the PEFT API at the linear layer that maps CNN features into the transformer's hidden space. This also makes `enable_input_require_grads()` install its requires_grad hook on the correct layer.

4. **`enable_input_require_grads()` is now called** (the previous version did not call it; combined with `gradient_checkpointing=True` this meant gradients may not have flowed correctly through the encoder LoRA adapters). With `mask_time_prob=0` the masking conflict that motivated the earlier omission does not occur.

5. **`modules_to_save=["lm_head"]`** — refines MMS's pretrained Maltese head on the MASRI read-speech domain. Unlike a randomly-initialised head, this is *adapting* a good starting point.


In [ ]:
from transformers import Wav2Vec2ForCTC
from peft import LoraConfig, get_peft_model

# Load with target_lang — attaches Maltese-specific lm_head and adapter
model = Wav2Vec2ForCTC.from_pretrained(
    model_id,
    target_lang=TGT_LANG,
    ignore_mismatched_sizes=True,
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    layerdrop=0.0,
    mask_time_prob=0.0,                       # disables SpecAugment in-place write conflict
    mask_feature_prob=0.0,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

# Freeze the CNN feature encoder (standard practice for Wav2Vec2-family)
model.freeze_feature_encoder()

# Satisfy PEFT/Trainer API — audio model has no token embeddings
model.get_input_embeddings = lambda: model.wav2vec2.feature_projection.projection

# Required with gradient_checkpointing + PEFT — gradients won't flow through LoRA adapters without this.
# Safe to call here because mask_time_prob=mask_feature_prob=0 disables the in-place masking that
# previously conflicted with the leaf-requires-grad tensor.
#model.enable_input_require_grads()

config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    modules_to_save=["lm_head"],              # refine pretrained Maltese head on MASRI domain
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/1095 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-1b-all
Key                        | Status     |                                                                                          
---------------------------+------------+------------------------------------------------------------------------------------------
wav2vec2.masked_spec_embed | UNEXPECTED |                                                                                          
lm_head.weight             | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([154, 1280]) vs model:torch.Size([83, 1280])
lm_head.bias               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([154]) vs model:torch.Size([83])            

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


trainable params: 7,970,643 || all params: 972,724,262 || trainable%: 0.8194


## 5. Trainer setup

`eval_dataset` is val; test is held out. Patience=5 because MMS converges slowly — the previous run was still improving at the 15-epoch ceiling, so the budget rather than convergence determined stopping.

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=model_save_dir,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,            # effective batch size 32 (matched across all 3 models)
    learning_rate=1e-3,                       # canonical MMS adapter LR (von Platen 2023; Pratap et al. 2024)
    weight_decay=0.005,
    warmup_steps=100,
    num_train_epochs=25,                      # raised from 15 — previous run hadn't converged at the old ceiling
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    logging_steps=50,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",              # evaluated on VAL, not test
    greater_is_better=False,
    remove_unused_columns=False,
    gradient_checkpointing=True,              # paired with enable_input_require_grads() above
    seed=42,
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=masri_dataset["train"],
    eval_dataset=masri_dataset["val"],        # VAL, not test
    processing_class=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

## 6. Zero-shot baseline on the test set

Especially important for MMS-1B-all because it ships with a pretrained Maltese adapter — this measures what the model can do *before* any LoRA training, so the LoRA contribution can be quantified.

In [ ]:
print("=== Zero-shot baseline (MMS-1B-all + pretrained Maltese adapter, no LoRA training) on TEST set ===")
zero_shot_metrics = trainer.evaluate(
    eval_dataset=masri_dataset["test"],
    metric_key_prefix="zero_shot",
)
for k, v in zero_shot_metrics.items():
    print(f"  {k}: {v}")

import json as _json
with open(os.path.join(model_save_dir, "zero_shot_test_metrics.json"), "w") as f:
    _json.dump({k: float(v) if isinstance(v, (int, float)) else v
                for k, v in zero_shot_metrics.items()}, f, indent=2)

=== Zero-shot baseline (MMS-1B-all + pretrained Maltese adapter, no LoRA training) on TEST set ===


early stopping required metric_for_best_model, but did not find eval_wer so early stopping is disabled


  zero_shot_loss: 0.9364521503448486
  zero_shot_model_preparation_time: 0.1039
  zero_shot_wer: 0.3380578433250335
  zero_shot_cer: 0.08885789555245623
  zero_shot_runtime: 68.0598
  zero_shot_samples_per_second: 7.317
  zero_shot_steps_per_second: 0.926


## 7. Training

In [ ]:
print("Starting MMS-1B-all LoRA fine-tuning (early stopping on val WER, patience=5)")
trainer.train()

Starting MMS-1B-all LoRA fine-tuning (early stopping on val WER, patience=5)


Epoch,Training Loss,Validation Loss,Model Preparation Time,Wer,Cer
1,1.382739,0.356262,0.103900,0.353567,0.083161
2,1.278038,0.340036,0.103900,0.352941,0.085034
3,1.110653,0.330667,0.103900,0.329996,0.079013
4,1.040428,0.311968,0.103900,0.313517,0.075600
5,0.858436,0.306393,0.103900,0.308928,0.074028
6,0.728705,0.316147,0.103900,0.306425,0.073393
7,0.670748,0.297804,0.103900,0.279725,0.067739
8,0.623211,0.328228,0.103900,0.293283,0.071620
9,0.518618,0.320325,0.103900,0.279307,0.070884
10,0.469692,0.326833,0.103900,0.270338,0.066134


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetunin

TrainOutput(global_step=3150, training_loss=0.4529152299865844, metrics={'train_runtime': 31172.7697, 'train_samples_per_second': 3.234, 'train_steps_per_second': 0.101, 'total_flos': 8.08041521039929e+19, 'train_loss': 0.4529152299865844, 'epoch': 25.0})

## 8. Save and final test-set evaluation

In [ ]:

trainer.save_model(model_save_dir)
processor.save_pretrained(model_save_dir)
print(f"Model saved to {model_save_dir}")


print("\n=== FINAL test-set evaluation (held-out, untouched during training) ===")
final_metrics = trainer.evaluate(
    eval_dataset=masri_dataset["test"],
    metric_key_prefix="final_test",
)
for k, v in final_metrics.items():
    print(f"  {k}: {v}")

with open(os.path.join(model_save_dir, "final_test_metrics.json"), "w") as f:
    _json.dump({k: float(v) if isinstance(v, (int, float)) else v
                for k, v in final_metrics.items()}, f, indent=2)
print(f"\nMetrics saved to {model_save_dir}/final_test_metrics.json")